# Feature Engineering — Final Pipeline

Clean consolidated notebook showing how all **61 features** used by the final LightGBM classifier (AUC 0.8629) are built.

**Input:** `dataset/merged_flights.parquet` — output of 04_master_merge.ipynb + 06_clustering.ipynb (must have `airline_cluster_label`)

**Output:** `dataset/FINAL_DATASET.parquet` — 61 model features + identifiers

## Pipeline
```
04_master_merge → merged_flights.parquet (77 cols)
06_clustering   → adds airline_cluster_label
This notebook   → adds 39 engineered features → saves FINAL_DATASET.parquet
```

In [8]:
import pandas as pd
import numpy as np
import time
import gc

flights = pd.read_parquet('dataset/merged_flights.parquet')
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])

# Build DEP_DATETIME needed for cascade and turnaround features
flights['DEP_DATETIME'] = flights['FL_DATE'] + pd.to_timedelta(
    (flights['CRS_DEP_TIME'] // 100) * 60 + (flights['CRS_DEP_TIME'] % 100), unit='m'
)

print(f'Loaded: {flights.shape[0]:,} rows x {flights.shape[1]} cols')
print(f'airline_cluster_label present: {"airline_cluster_label" in flights.columns}')

Loaded: 18,227,796 rows x 79 cols
airline_cluster_label present: True


## 1. Financial Features

`profit_margin` from BTS Form 41 financials.

In [9]:
flights['profit_margin'] = flights['OP_PROFIT_LOSS'] / flights['OP_REVENUES']
print(f'profit_margin nulls: {flights["profit_margin"].isna().sum():,}')

profit_margin nulls: 0


## 2. Weather Severity Composites

Weighted combination of wind speed, precipitation, and sky condition.

In [10]:
wind_norm_origin = (flights['origin_wind_speed'] / 30).clip(0, 1)
precip_norm_origin = (flights['origin_precip_1hr'] / 5).clip(0, 1)
sky_norm_origin = flights['origin_sky_condition'] / 9

flights['origin_weather_severity'] = (
    wind_norm_origin * 0.4 + precip_norm_origin * 0.3 + sky_norm_origin * 0.3
).astype('float32')

wind_norm_dest = (flights['dest_wind_speed'] / 30).clip(0, 1)
precip_norm_dest = (flights['dest_precip_1hr'] / 5).clip(0, 1)
sky_norm_dest = flights['dest_sky_condition'] / 9

flights['dest_weather_severity'] = (
    wind_norm_dest * 0.4 + precip_norm_dest * 0.3 + sky_norm_dest * 0.3
).astype('float32')

print(f'origin_weather_severity nulls: {flights["origin_weather_severity"].isna().sum():,}')

origin_weather_severity nulls: 8,810,842


## 3. Cascade Features

6-hour lookback window per aircraft tail. Counts prior delayed flights and accumulates their delay minutes.

In [11]:
def compute_cascade(tail_df):
    tail_df = tail_df.sort_values('DEP_DATETIME').copy()
    n = len(tail_df)
    cascade_score         = np.zeros(n, dtype='int8')
    cascade_delay_minutes = np.zeros(n, dtype='float32')
    hours_since_last      = np.full(n, np.nan, dtype='float32')
    dep_times  = tail_df['DEP_DATETIME'].values
    delayed    = tail_df['ARR_DEL15'].values
    delay_mins = tail_df['ARR_DELAY'].values
    for i in range(1, n):
        window_start = dep_times[i] - np.timedelta64(6, 'h')
        for j in range(i-1, -1, -1):
            if dep_times[j] < window_start:
                break
            if delayed[j] == 1.0:
                cascade_score[i] += 1
                cascade_delay_minutes[i] += max(0, delay_mins[j])
                if np.isnan(hours_since_last[i]):
                    hours_since_last[i] = (
                        dep_times[i] - dep_times[j]
                    ) / np.timedelta64(1, 'h')
    tail_df['cascade_score']          = cascade_score
    tail_df['cascade_delay_minutes']  = cascade_delay_minutes
    tail_df['hours_since_last_delay'] = hours_since_last
    return tail_df

print('Running cascade on full 18M rows...')
start = time.time()
cascade_results = flights[['TAIL_NUM', 'DEP_DATETIME', 'ARR_DEL15', 'ARR_DELAY']].copy()
cascade_results = cascade_results.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)
elapsed = time.time() - start
print(f'Done in {elapsed:.1f} seconds')

cascade_to_join = cascade_results[['cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']]
flights = flights.join(cascade_to_join, how='left')
print(f'Shape: {flights.shape}')

Running cascade on full 18M rows...


/var/folders/_r/scvy88yj1wl9m4x325v7q1200000gn/T/ipykernel_58295/1627726466.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cascade_results = cascade_results.groupby('TAIL_NUM', group_keys=False).apply(compute_cascade)


Done in 54.3 seconds
Shape: (18227796, 85)


## 4. Rolling Delay Rates (30d and 7d)

All point-in-time safe — uses shift(1) before rolling to prevent leakage.

In [12]:
# airline_delay_rate_30d
carrier_daily = flights.groupby(['OP_UNIQUE_CARRIER', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
carrier_daily = carrier_daily.sort_values(['OP_UNIQUE_CARRIER', 'FL_DATE'])
carrier_daily['airline_delay_rate_30d'] = carrier_daily.groupby('OP_UNIQUE_CARRIER')['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(carrier_daily[['OP_UNIQUE_CARRIER', 'FL_DATE', 'airline_delay_rate_30d']], on=['OP_UNIQUE_CARRIER', 'FL_DATE'], how='left')
print(f'airline_delay_rate_30d: {flights["airline_delay_rate_30d"].isna().sum():,} nulls')

# origin_delay_rate_30d
origin_daily = flights.groupby(['ORIGIN', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
origin_daily = origin_daily.sort_values(['ORIGIN', 'FL_DATE'])
origin_daily['origin_delay_rate_30d'] = origin_daily.groupby('ORIGIN')['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(origin_daily[['ORIGIN', 'FL_DATE', 'origin_delay_rate_30d']], on=['ORIGIN', 'FL_DATE'], how='left')
print(f'origin_delay_rate_30d: {flights["origin_delay_rate_30d"].isna().sum():,} nulls')

# dest_delay_rate_30d
dest_daily = flights.groupby(['DEST', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
dest_daily = dest_daily.sort_values(['DEST', 'FL_DATE'])
dest_daily['dest_delay_rate_30d'] = dest_daily.groupby('DEST')['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(dest_daily[['DEST', 'FL_DATE', 'dest_delay_rate_30d']], on=['DEST', 'FL_DATE'], how='left')
print(f'dest_delay_rate_30d: {flights["dest_delay_rate_30d"].isna().sum():,} nulls')

# route_delay_rate_30d
route_daily = flights.groupby(['ORIGIN', 'DEST', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
route_daily = route_daily.sort_values(['ORIGIN', 'DEST', 'FL_DATE'])
route_daily['route_delay_rate_30d'] = route_daily.groupby(['ORIGIN', 'DEST'])['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(route_daily[['ORIGIN', 'DEST', 'FL_DATE', 'route_delay_rate_30d']], on=['ORIGIN', 'DEST', 'FL_DATE'], how='left')
print(f'route_delay_rate_30d: {flights["route_delay_rate_30d"].isna().sum():,} nulls')

# origin_avg_taxi_out_30d
taxi_daily = flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])['TAXI_OUT'].mean().reset_index().rename(columns={'TAXI_OUT': 'daily_avg_taxi'})
taxi_daily = taxi_daily.sort_values(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])
taxi_daily['origin_avg_taxi_out_30d'] = taxi_daily.groupby(['ORIGIN', 'DEP_HOUR'])['daily_avg_taxi'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(taxi_daily[['ORIGIN', 'FL_DATE', 'DEP_HOUR', 'origin_avg_taxi_out_30d']], on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')
print(f'origin_avg_taxi_out_30d: {flights["origin_avg_taxi_out_30d"].isna().sum():,} nulls')

gc.collect()

airline_delay_rate_30d: 114,775 nulls
origin_delay_rate_30d: 115,197 nulls
dest_delay_rate_30d: 115,210 nulls
route_delay_rate_30d: 133,471 nulls
origin_avg_taxi_out_30d: 132,093 nulls


0

In [13]:
# flight_num_delay_rate_30d
flight_daily = flights.groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
flight_daily = flight_daily.sort_values(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'])
flight_daily['flight_num_delay_rate_30d'] = flight_daily.groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM'])['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(flight_daily[['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE', 'flight_num_delay_rate_30d']], on=['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'], how='left')
print(f'flight_num_delay_rate_30d: {flights["flight_num_delay_rate_30d"].isna().sum():,} nulls')

# origin_hour_delay_rate_30d
origin_hour_daily = flights.groupby(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
origin_hour_daily = origin_hour_daily.sort_values(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])
origin_hour_daily['origin_hour_delay_rate_30d'] = origin_hour_daily.groupby(['ORIGIN', 'DEP_HOUR'])['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(origin_hour_daily[['ORIGIN', 'DEP_HOUR', 'FL_DATE', 'origin_hour_delay_rate_30d']], on=['ORIGIN', 'DEP_HOUR', 'FL_DATE'], how='left')
print(f'origin_hour_delay_rate_30d: {flights["origin_hour_delay_rate_30d"].isna().sum():,} nulls')

# carrier_origin_delay_rate_30d
carrier_origin_daily = flights.groupby(['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
carrier_origin_daily = carrier_origin_daily.sort_values(['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'])
carrier_origin_daily['carrier_origin_delay_rate_30d'] = carrier_origin_daily.groupby(['OP_UNIQUE_CARRIER', 'ORIGIN'])['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(carrier_origin_daily[['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE', 'carrier_origin_delay_rate_30d']], on=['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'], how='left')
print(f'carrier_origin_delay_rate_30d: {flights["carrier_origin_delay_rate_30d"].isna().sum():,} nulls')

# dest_hour_delay_rate_30d
dest_hour_daily = flights.groupby(['DEST', 'ARR_HOUR', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
dest_hour_daily = dest_hour_daily.sort_values(['DEST', 'ARR_HOUR', 'FL_DATE'])
dest_hour_daily['dest_hour_delay_rate_30d'] = dest_hour_daily.groupby(['DEST', 'ARR_HOUR'])['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
flights = flights.merge(dest_hour_daily[['DEST', 'ARR_HOUR', 'FL_DATE', 'dest_hour_delay_rate_30d']], on=['DEST', 'ARR_HOUR', 'FL_DATE'], how='left')
print(f'dest_hour_delay_rate_30d: {flights["dest_hour_delay_rate_30d"].isna().sum():,} nulls')

# airline_delay_rate_7d
carrier_daily7 = flights.groupby(['OP_UNIQUE_CARRIER', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
carrier_daily7 = carrier_daily7.sort_values(['OP_UNIQUE_CARRIER', 'FL_DATE'])
carrier_daily7['airline_delay_rate_7d'] = carrier_daily7.groupby('OP_UNIQUE_CARRIER')['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=7, min_periods=3).mean())
flights = flights.merge(carrier_daily7[['OP_UNIQUE_CARRIER', 'FL_DATE', 'airline_delay_rate_7d']], on=['OP_UNIQUE_CARRIER', 'FL_DATE'], how='left')
print(f'airline_delay_rate_7d: {flights["airline_delay_rate_7d"].isna().sum():,} nulls')

# origin_delay_rate_7d
origin_daily7 = flights.groupby(['ORIGIN', 'FL_DATE'])['ARR_DEL15'].mean().reset_index().rename(columns={'ARR_DEL15': 'daily_delay_rate'})
origin_daily7 = origin_daily7.sort_values(['ORIGIN', 'FL_DATE'])
origin_daily7['origin_delay_rate_7d'] = origin_daily7.groupby('ORIGIN')['daily_delay_rate'].transform(lambda x: x.shift(1).rolling(window=7, min_periods=3).mean())
flights = flights.merge(origin_daily7[['ORIGIN', 'FL_DATE', 'origin_delay_rate_7d']], on=['ORIGIN', 'FL_DATE'], how='left')
print(f'origin_delay_rate_7d: {flights["origin_delay_rate_7d"].isna().sum():,} nulls')

gc.collect()

flight_num_delay_rate_30d: 291,662 nulls
origin_hour_delay_rate_30d: 132,093 nulls
carrier_origin_delay_rate_30d: 119,862 nulls
dest_hour_delay_rate_30d: 133,347 nulls
airline_delay_rate_7d: 49,432 nulls
origin_delay_rate_7d: 49,625 nulls


0

## 5. Congestion + Turnaround Features

In [14]:
# hourly_flight_count
hourly_congestion = flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR']).size().reset_index(name='hourly_flight_count')
flights = flights.merge(hourly_congestion, on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')
print(f'hourly_flight_count nulls: {flights["hourly_flight_count"].isna().sum()}')

# dest_hourly_flight_count
dest_hourly = flights.groupby(['DEST', 'FL_DATE', 'ARR_HOUR']).size().reset_index(name='dest_hourly_flight_count')
flights = flights.merge(dest_hourly, on=['DEST', 'FL_DATE', 'ARR_HOUR'], how='left')
print(f'dest_hourly_flight_count nulls: {flights["dest_hourly_flight_count"].isna().sum()}')

# tail_flight_num_today
flights_sorted = flights.sort_values(['TAIL_NUM', 'DEP_DATETIME'])
flights['tail_flight_num_today'] = flights_sorted.groupby(['TAIL_NUM', 'FL_DATE']).cumcount().astype('int8')
print(f'tail_flight_num_today max: {flights["tail_flight_num_today"].max()}')

# scheduled_turnaround_buffer
flights_sorted2 = flights[['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'DEP_DATETIME']].copy()
flights_sorted2 = flights_sorted2.sort_values(['TAIL_NUM', 'DEP_DATETIME'])
flights_sorted2['prev_crs_arr'] = flights_sorted2.groupby('TAIL_NUM')['CRS_ARR_TIME'].shift(1)
flights_sorted2['prev_fl_date'] = flights_sorted2.groupby('TAIL_NUM')['FL_DATE'].shift(1)
mask = flights_sorted2['prev_crs_arr'].notna()
hours = (flights_sorted2.loc[mask, 'prev_crs_arr'] // 100).astype(int)
mins = (flights_sorted2.loc[mask, 'prev_crs_arr'] % 100).astype(int)
flights_sorted2.loc[mask, 'prev_arr_datetime'] = pd.to_datetime(flights_sorted2.loc[mask, 'prev_fl_date']) + pd.to_timedelta(hours * 60 + mins, unit='m')
flights_sorted2['scheduled_turnaround_buffer'] = ((flights_sorted2['DEP_DATETIME'] - flights_sorted2['prev_arr_datetime']).dt.total_seconds() / 60).astype('float32')
flights_sorted2.loc[(flights_sorted2['scheduled_turnaround_buffer'] < 0) | (flights_sorted2['scheduled_turnaround_buffer'] > 360), 'scheduled_turnaround_buffer'] = np.nan
flights = flights.join(flights_sorted2[['scheduled_turnaround_buffer']], how='left')
print(f'scheduled_turnaround_buffer nulls: {flights["scheduled_turnaround_buffer"].isna().sum():,}')

hourly_flight_count nulls: 0
dest_hourly_flight_count nulls: 0
tail_flight_num_today max: 13
scheduled_turnaround_buffer nulls: 4,937,387


## 6. Inbound Arrival Delay (3h lookback)

Average arrival delay at origin/destination over the prior 3 hours.

In [15]:
# inbound_arr_delay_3h
arrivals = flights[['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy().rename(columns={'DEST': 'AIRPORT'})
arr_hourly = arrivals.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index().rename(columns={'ARR_DELAY': 'hourly_arr_delay'})
arr_hourly = arr_hourly.sort_values(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])
results = []
for hour_offset in range(3):
    temp = arr_hourly.copy()
    temp['DEP_HOUR'] = temp['ARR_HOUR'] + hour_offset + 1
    temp = temp[temp['DEP_HOUR'] <= 23]
    results.append(temp[['AIRPORT', 'FL_DATE', 'DEP_HOUR', 'hourly_arr_delay']])
combined = pd.concat(results)
inbound_3h = combined.groupby(['AIRPORT', 'FL_DATE', 'DEP_HOUR'])['hourly_arr_delay'].mean().reset_index().rename(columns={'AIRPORT': 'ORIGIN', 'hourly_arr_delay': 'inbound_arr_delay_3h'})
flights = flights.merge(inbound_3h, on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')
print(f'inbound_arr_delay_3h nulls: {flights["inbound_arr_delay_3h"].isna().sum():,}')

# dest_inbound_arr_delay_3h
dest_arrivals = flights[['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy().rename(columns={'DEST': 'AIRPORT'})
dest_arr_hourly = dest_arrivals.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index().rename(columns={'ARR_DELAY': 'hourly_arr_delay'})
results2 = []
for hour_offset in range(3):
    temp = dest_arr_hourly.copy()
    temp['ARR_HOUR_lookup'] = temp['ARR_HOUR'] + hour_offset + 1
    temp = temp[temp['ARR_HOUR_lookup'] <= 23]
    results2.append(temp[['AIRPORT', 'FL_DATE', 'ARR_HOUR_lookup', 'hourly_arr_delay']])
combined2 = pd.concat(results2)
dest_inbound = combined2.groupby(['AIRPORT', 'FL_DATE', 'ARR_HOUR_lookup'])['hourly_arr_delay'].mean().reset_index().rename(columns={'AIRPORT': 'DEST', 'ARR_HOUR_lookup': 'ARR_HOUR', 'hourly_arr_delay': 'dest_inbound_arr_delay_3h'})
flights = flights.merge(dest_inbound, on=['DEST', 'FL_DATE', 'ARR_HOUR'], how='left')
print(f'dest_inbound_arr_delay_3h nulls: {flights["dest_inbound_arr_delay_3h"].isna().sum():,}')

inbound_arr_delay_3h nulls: 1,830,425
dest_inbound_arr_delay_3h nulls: 966,910


## 7. Previous Tail Arrival Delay

Arrival delay of the previous flight of this aircraft. Set to NaN if gap > 6 hours.

In [16]:
flights_sorted3 = flights[['TAIL_NUM', 'DEP_DATETIME', 'ARR_DELAY']].copy()
flights_sorted3 = flights_sorted3.sort_values(['TAIL_NUM', 'DEP_DATETIME'])
flights_sorted3['prev_tail_arr_delay'] = flights_sorted3.groupby('TAIL_NUM')['ARR_DELAY'].shift(1)
flights_sorted3['prev_dep'] = flights_sorted3.groupby('TAIL_NUM')['DEP_DATETIME'].shift(1)
gap_hours = (flights_sorted3['DEP_DATETIME'] - flights_sorted3['prev_dep']).dt.total_seconds() / 3600
flights_sorted3.loc[gap_hours > 6, 'prev_tail_arr_delay'] = np.nan
flights = flights.join(flights_sorted3[['prev_tail_arr_delay']], how='left')
print(f'prev_tail_arr_delay nulls: {flights["prev_tail_arr_delay"].isna().sum():,}')

# Reset index so subsequent operations align correctly
flights = flights.reset_index(drop=True)

prev_tail_arr_delay nulls: 5,705,616


## 8. National Hub Delay (2h lookback)

Average arrival delay at top 10 hub airports over prior 2 hours.

In [17]:
top10_hubs = ['ATL', 'DEN', 'DFW', 'ORD', 'LAX', 'CLT', 'MCO', 'LAS', 'PHX', 'MIA']
hub_arrivals = flights[flights['DEST'].isin(top10_hubs)][['DEST', 'FL_DATE', 'ARR_HOUR', 'ARR_DELAY']].copy()
hub_hourly = hub_arrivals.groupby(['FL_DATE', 'ARR_HOUR'])['ARR_DELAY'].mean().reset_index().rename(columns={'ARR_DELAY': 'hub_hourly_delay'})
hub_hourly = hub_hourly.sort_values(['FL_DATE', 'ARR_HOUR'])
results3 = []
for offset in range(2):
    temp = hub_hourly.copy()
    temp['DEP_HOUR'] = temp['ARR_HOUR'] + offset + 1
    temp = temp[temp['DEP_HOUR'] <= 23]
    results3.append(temp[['FL_DATE', 'DEP_HOUR', 'hub_hourly_delay']])
combined3 = pd.concat(results3)
hub_fever = combined3.groupby(['FL_DATE', 'DEP_HOUR'])['hub_hourly_delay'].mean().reset_index().rename(columns={'hub_hourly_delay': 'national_hub_delay_2h'})
flights = flights.merge(hub_fever, on=['FL_DATE', 'DEP_HOUR'], how='left')
print(f'national_hub_delay_2h nulls: {flights["national_hub_delay_2h"].isna().sum():,}')

national_hub_delay_2h nulls: 30,864


## 9. Weather Delta Features (3h change)

Merged from NOAA ISD-Lite via datetime key. Captures approaching storm fronts.

In [18]:
flights['origin_weather_dt'] = flights['FL_DATE'] + pd.to_timedelta(flights['DEP_HOUR'], unit='h')
flights['dest_weather_dt']   = flights['FL_DATE'] + pd.to_timedelta(flights['ARR_HOUR'], unit='h')

# Load and clean weather — sentinel values first
weather = pd.read_parquet('dataset/weather/weather_clean.parquet')
weather['DATETIME'] = pd.to_datetime(weather['DATETIME'])
weather = weather.sort_values(['AIRPORT', 'DATETIME']).reset_index(drop=True)

weather.loc[weather['PRESSURE'].isin([0, 9999, 999.9, 99.9]) | (weather['PRESSURE'] < 900), 'PRESSURE'] = np.nan
weather.loc[weather['WIND_SPEED'].isin([99.9, 999.9]), 'WIND_SPEED'] = np.nan
weather.loc[weather['TEMP'].isin([999.9, 99.9]), 'TEMP'] = np.nan

# Compute 3h deltas per airport
weather = weather.set_index('DATETIME')
deltas = []
for airport, grp in weather.groupby('AIRPORT'):
    grp = grp.sort_index()
    grp['pressure_delta_3h']   = grp['PRESSURE']   - grp['PRESSURE'].shift(3)
    grp['wind_speed_delta_3h'] = grp['WIND_SPEED']  - grp['WIND_SPEED'].shift(3)
    grp['temp_delta_3h']       = grp['TEMP']        - grp['TEMP'].shift(3)
    deltas.append(grp[['AIRPORT', 'pressure_delta_3h', 'wind_speed_delta_3h', 'temp_delta_3h']].reset_index())

weather_deltas = pd.concat(deltas, ignore_index=True)
weather_deltas['DATETIME'] = pd.to_datetime(weather_deltas['DATETIME'])

# Merge to flights
origin_deltas = weather_deltas.rename(columns={'AIRPORT': 'ORIGIN', 'DATETIME': 'origin_weather_dt', 'pressure_delta_3h': 'origin_pressure_delta_3h', 'wind_speed_delta_3h': 'origin_wind_speed_delta_3h', 'temp_delta_3h': 'origin_temp_delta_3h'})
dest_deltas   = weather_deltas.rename(columns={'AIRPORT': 'DEST',   'DATETIME': 'dest_weather_dt',   'pressure_delta_3h': 'dest_pressure_delta_3h',   'wind_speed_delta_3h': 'dest_wind_speed_delta_3h',   'temp_delta_3h': 'dest_temp_delta_3h'})

flights = flights.merge(origin_deltas[['ORIGIN', 'origin_weather_dt', 'origin_pressure_delta_3h', 'origin_wind_speed_delta_3h']], on=['ORIGIN', 'origin_weather_dt'], how='left')
flights = flights.merge(dest_deltas[['DEST', 'dest_weather_dt', 'dest_pressure_delta_3h', 'dest_wind_speed_delta_3h']],         on=['DEST',   'dest_weather_dt'],   how='left')

for col in ['origin_pressure_delta_3h', 'dest_pressure_delta_3h', 'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h']:
    print(f'  {col}: {flights[col].notna().mean()*100:.1f}% matched')

del weather, weather_deltas, origin_deltas, dest_deltas
gc.collect()

  origin_pressure_delta_3h: 89.1% matched
  dest_pressure_delta_3h: 89.0% matched
  origin_wind_speed_delta_3h: 91.2% matched
  dest_wind_speed_delta_3h: 91.1% matched


0

## 10. Time + Hourly Delay Rate Features

In [19]:
flights['day_of_year'] = flights['FL_DATE'].dt.dayofyear

# origin_dep_delay_rate_1h and dest_dep_delay_rate_1h (uses DEP_DEL15, shift per group)
hourly_origin2 = flights.groupby(['ORIGIN', 'FL_DATE', 'DEP_HOUR']).agg(dep_delayed=('DEP_DEL15', 'sum'), dep_total=('DEP_DEL15', 'count')).reset_index()
hourly_origin2 = hourly_origin2.sort_values(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])
hourly_origin2['prev_hr_delayed'] = hourly_origin2.groupby('ORIGIN')['dep_delayed'].shift(1)
hourly_origin2['prev_hr_total']   = hourly_origin2.groupby('ORIGIN')['dep_total'].shift(1)
hourly_origin2['origin_dep_delay_rate_1h'] = hourly_origin2['prev_hr_delayed'] / hourly_origin2['prev_hr_total']
flights = flights.merge(hourly_origin2[['ORIGIN', 'FL_DATE', 'DEP_HOUR', 'origin_dep_delay_rate_1h']], on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')

hourly_dest2 = flights.groupby(['DEST', 'FL_DATE', 'DEP_HOUR']).agg(dep_delayed=('DEP_DEL15', 'sum'), dep_total=('DEP_DEL15', 'count')).reset_index()
hourly_dest2 = hourly_dest2.sort_values(['DEST', 'FL_DATE', 'DEP_HOUR'])
hourly_dest2['prev_hr_delayed'] = hourly_dest2.groupby('DEST')['dep_delayed'].shift(1)
hourly_dest2['prev_hr_total']   = hourly_dest2.groupby('DEST')['dep_total'].shift(1)
hourly_dest2['dest_dep_delay_rate_1h'] = hourly_dest2['prev_hr_delayed'] / hourly_dest2['prev_hr_total']
flights = flights.merge(hourly_dest2[['DEST', 'FL_DATE', 'DEP_HOUR', 'dest_dep_delay_rate_1h']], on=['DEST', 'FL_DATE', 'DEP_HOUR'], how='left')

print(f'origin_dep_delay_rate_1h nulls: {flights["origin_dep_delay_rate_1h"].isna().sum():,}')
print(f'dest_dep_delay_rate_1h nulls: {flights["dest_dep_delay_rate_1h"].isna().sum():,}')

origin_dep_delay_rate_1h nulls: 688
dest_dep_delay_rate_1h nulls: 701


## 11. Composite Stress + Fatigue Features

In [20]:
# origin_stress_index
flights['origin_stress_index'] = flights['origin_dep_delay_rate_1h'] * flights['hourly_flight_count']

# real_time_turn_gap — top SHAP feature (mean |SHAP| 0.773)
flights['real_time_turn_gap'] = flights['scheduled_turnaround_buffer'] - flights['prev_tail_arr_delay']

# tail_delays_today
flights = flights.sort_values(['TAIL_NUM', 'FL_DATE', 'DEP_HOUR'])
flights['tail_delays_today'] = flights.groupby(['TAIL_NUM', 'FL_DATE'])['ARR_DEL15'].cumsum().shift(1).fillna(0)

# tail_active_hours
first_dep = flights.groupby(['TAIL_NUM', 'FL_DATE'])['DEP_HOUR'].transform('first')
flights['tail_active_hours'] = flights['DEP_HOUR'] - first_dep

# origin_pressure_drop_stress
flights['origin_pressure_drop_stress'] = flights['origin_pressure_delta_3h'].clip(upper=0).abs() * flights['hourly_flight_count']

# airport_fatigue_index
flights = flights.sort_values(['ORIGIN', 'FL_DATE', 'DEP_HOUR'])
flights['airport_fatigue_index'] = flights.groupby(['ORIGIN', 'FL_DATE'])['DEP_DEL15'].cumsum().shift(1).fillna(0)

print('All composite features added')
print(f'real_time_turn_gap % negative: {(flights["real_time_turn_gap"] < 0).mean()*100:.1f}%')

All composite features added
real_time_turn_gap % negative: 3.9%


In [21]:
from pandas.tseries.holiday import USFederalHolidayCalendar

cal = USFederalHolidayCalendar()
holidays = cal.holidays(start='2023-01-01', end='2025-12-31')

holiday_range = set()
for h in holidays:
    for offset in range(-2, 3):
        holiday_range.add(h + pd.Timedelta(days=offset))

flights['IS_HOLIDAY'] = flights['FL_DATE'].isin(holiday_range).astype('int8')
print(f'IS_HOLIDAY rate: {flights["IS_HOLIDAY"].mean():.3f}')

IS_HOLIDAY rate: 0.138


## 12. Verify All 61 Features Present

In [22]:
FINAL_FEATURES = [
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY', 'DISTANCE',
    'profit_margin', 'origin_monthly_passengers',
    'origin_temp', 'origin_dew_point', 'origin_pressure', 'origin_wind_dir',
    'origin_wind_speed', 'origin_precip_1hr', 'origin_weather_severity',
    'dest_temp', 'dest_dew_point', 'dest_pressure', 'dest_wind_dir',
    'dest_wind_speed', 'dest_precip_1hr', 'dest_weather_severity',
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    'carrier_origin_delay_rate_30d', 'dest_hour_delay_rate_30d',
    'airline_delay_rate_7d', 'origin_delay_rate_7d',
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    'hourly_flight_count', 'scheduled_turnaround_buffer', 'tail_flight_num_today',
    'dest_hourly_flight_count',
    'inbound_arr_delay_3h', 'dest_inbound_arr_delay_3h',
    'prev_tail_arr_delay', 'national_hub_delay_2h',
    'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'airline_cluster_label',
    'origin_pressure_delta_3h', 'dest_pressure_delta_3h',
    'origin_wind_speed_delta_3h', 'dest_wind_speed_delta_3h',
    'day_of_year', 'origin_dep_delay_rate_1h', 'dest_dep_delay_rate_1h',
    'origin_stress_index', 'real_time_turn_gap',
    'tail_delays_today', 'tail_active_hours',
    'origin_pressure_drop_stress', 'airport_fatigue_index'
]

assert len(FINAL_FEATURES) == 61
missing = [f for f in FINAL_FEATURES if f not in flights.columns]
print(f'Expected: 61 | Present: {61 - len(missing)} | Missing: {len(missing)}')
if missing:
    for f in missing:
        print(f'  MISSING: {f}')
else:
    print('All 61 features present ✓')

Expected: 61 | Present: 61 | Missing: 0
All 61 features present ✓


## 13. Save Final Dataset

In [23]:
flights.to_parquet('dataset/FINAL_DATASET.parquet', index=False, engine='pyarrow', compression='snappy')

import os
size_mb = os.path.getsize('dataset/FINAL_DATASET.parquet') / 1024 / 1024
print(f'Saved: {flights.shape[0]:,} rows x {flights.shape[1]} cols')
print(f'Size: {size_mb:.1f} MB')

Saved: 18,227,796 rows x 120 cols
Size: 1548.0 MB
